# VLM-DENTAL - Trace Generation & YOLO Grounding Workspace

This notebook handles heavy dataset downloading, autonomous CoT trace generation, YOLO grounding tool training with 5-fold cross-validation, downloading final models/traces, and pushing generated artifacts to GitHub.

## 1. Environment Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**(Optional) Fresh Start Cleanup:**
Run this cell ONLY if you need to completely delete the VLM-DENTAL folder from your Google Drive to start over.

In [ ]:
# Uncomment the line below to delete the folder, then run the cell
# !rm -rf /content/drive/MyDrive/VLM-DENTAL

In [ ]:
import os

# Set this to True to save the 10GB dataset and code to Google Drive.
# Set this to False to download them to temporary Colab storage (faster, but lost when Colab disconnects).
# NOTE: Regardless of this setting, your generated traces and YOLO weights will ALWAYS be saved to Google Drive.
SAVE_DATASET_AND_CODE_TO_DRIVE = True

drive_path = "/content/drive/MyDrive/VLM-DENTAL"
colab_path = "/content/VLM-DENTAL"
work_dir = drive_path if SAVE_DATASET_AND_CODE_TO_DRIVE else colab_path

In [ ]:
import os

if SAVE_DATASET_AND_CODE_TO_DRIVE:
    os.chdir("/content/drive/MyDrive")
else:
    os.chdir("/content")

if not os.path.exists("VLM-DENTAL"):
    os.system("git clone https://github.com/rezaxr14/VLM-DENTAL.git")

os.chdir(work_dir)
os.system("git pull")

# Ensure output directories exist in Drive
os.makedirs(f"{drive_path}/data/traces", exist_ok=True)
os.makedirs(f"{drive_path}/data/models", exist_ok=True)

In [ ]:
# Install the project and all its requirements
!pip install -e .
!pip install python-dotenv pandas pillow google-generativeai anthropic huggingface_hub ultralytics

## 2. Configure Credentials (Colab Secrets Tab Support)
Loads API keys and GitHub tokens automatically from Colab Secrets tab (`google.colab.userdata`).

In [ ]:
import os

# 1. Attempt loading keys from Google Colab Secrets (userdata tab)
try:
    from google.colab import userdata
    
    # Gemini API Keys
    try:
        gemini_keys = userdata.get('GEMINI_API_KEYS')
        if gemini_keys:
            os.environ['GEMINI_API_KEYS'] = gemini_keys
    except Exception:
        try:
            gemini_key = userdata.get('GEMINI_API_KEY')
            if gemini_key:
                os.environ['GEMINI_API_KEYS'] = gemini_key
        except Exception:
            pass

    # Hugging Face Token
    try:
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            os.environ['HF_TOKEN'] = hf_token
    except Exception:
        pass

    # GitHub Access Token
    try:
        gh_token = userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN')
        if gh_token:
            os.environ['GITHUB_TOKEN'] = gh_token
            os.environ['GH_TOKEN'] = gh_token
    except Exception:
        pass

    # Git User Info
    try:
        git_name = userdata.get('GIT_USER_NAME')
        if git_name:
            os.environ['GIT_USER_NAME'] = git_name
    except Exception:
        pass

    try:
        git_email = userdata.get('GIT_USER_EMAIL')
        if git_email:
            os.environ['GIT_USER_EMAIL'] = git_email
    except Exception:
        pass

    print("Secrets tab checked.")
except ImportError:
    print("Not running in Colab environment or userdata API unavailable.")

# 2. Fallbacks for manual entry if secrets are not set in the tab
if 'GEMINI_API_KEYS' not in os.environ or os.environ['GEMINI_API_KEYS'].startswith('YOUR_'):
    os.environ['GEMINI_API_KEYS'] = 'YOUR_API_KEY_1,YOUR_API_KEY_2'

if 'HF_TOKEN' not in os.environ or os.environ['HF_TOKEN'].startswith('YOUR_'):
    os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'

if 'GITHUB_TOKEN' not in os.environ:
    os.environ['GITHUB_TOKEN'] = 'YOUR_GITHUB_TOKEN_HERE'

# Status Report
print("--- Credentials Status ---")
print(f"Gemini API Key set: {'Yes' if os.environ.get('GEMINI_API_KEYS') and not os.environ['GEMINI_API_KEYS'].startswith('YOUR_') else 'No (using placeholder)'}")
print(f"Hugging Face Token set: {'Yes' if os.environ.get('HF_TOKEN') and not os.environ['HF_TOKEN'].startswith('YOUR_') else 'No (using placeholder)'}")
print(f"GitHub Token set: {'Yes' if os.environ.get('GITHUB_TOKEN') and not os.environ['GITHUB_TOKEN'].startswith('YOUR_') else 'No (using placeholder)'}")

## 3. Dataset Download & Cleanup
Run this to download the dataset if you haven't already. It will extract and structure it automatically.

In [ ]:
!python download_and_cleanup.py

In [ ]:
# Delete unused partial datasets to save space
!rm -rf data/dentex/DENTEX/training_data/disease
!rm -rf data/dentex/DENTEX/training_data/quadrant
!rm -rf data/dentex/DENTEX/training_data/unlabelled/

!rm -rf data/dentex/DENTEX/testing_data/disease
!rm -rf data/dentex/DENTEX/testing_data/quadrant

# Delete corrupted cache folder from any previous bugs (if it exists)
!rm -rf "C:\\Users\\rezax\\dental_agent_cache"

## 4. Autonomous CoT Trace Generation
Runs the daily trace generator and continuously saves progress to `train_cot_traces.jsonl`.

In [ ]:
!python scripts/run_daily_trace_generator.py --split train --output /content/drive/MyDrive/VLM-DENTAL/data/traces/train_cot_traces.jsonl

## 5. YOLO Grounding Tool — 5-Fold Cross-Validation
Convert DENTEX annotations to YOLO format with 5-fold CV splits. Each fold trains independently from scratch, then the best fold's weights are selected as the final grounding model. The 50 validation images are held out permanently and never used in any fold's training.

In [ ]:
# Convert COCO annotations to YOLO format with 5-fold cross-validation splits
# - Training pool (1339 images) is split into 5 folds via KFold
# - 50 validation images are held out permanently as a separate test set
!python scripts/prepare_yolo_dataset.py --mode cv --folds 5 

In [ ]:
# Training config
RESUME = True  # Set True to continue training from last checkpoint (safe to leave True)

# Build command — only pass --resume if there is a checkpoint to resume from
resume_flag = "--resume" if RESUME else ""
!python scripts/train_grounding_tool.py --cross-validate --model yolov8m.pt --epochs 60 --patience 20 --batch 16 --device 0 $resume_flag

In [ ]:
import json
from pathlib import Path

results_path = Path("data/models/grounding_tool_cv_best/cv_results.json")
if results_path.exists():
    results = json.loads(results_path.read_text())
    print(f"{'Fold':<6} {'mAP50':<10} {'mAP50-95':<10} {'Precision':<10} {'Recall':<10}")
    print("-" * 46)
    for r in results["folds"]:
        marker = " <-- BEST" if r["fold"] == results["best_fold"] else ""
        print(f"{r['fold']:<6} {r['map50']:<10.4f} {r['map50_95']:<10.4f} {r['precision']:<10.4f} {r['recall']:<10.4f}{marker}")
    print("-" * 46)
    print(f"Mean:  {results['mean_map50']:.4f} +/- {results['std_map50']:.4f}   {results['mean_map50_95']:.4f} +/- {results['std_map50_95']:.4f}")
    print(f"\nBest fold: {results['best_fold']}")
    print(f"Best weights: data/models/grounding_tool_cv_best/weights/best.pt")
else:
    print("CV results not found. Run the cross-validation training cell first.")

## 6. Download Generated Models & Traces Locally
Triggers interactive browser download of generated CoT traces and trained YOLO weights (`best.pt`).

In [ ]:
import os
import shutil

# Options
DOWNLOAD_TRACES = True
DOWNLOAD_CV_BEST = True       # Best fold weights (final model)
DOWNLOAD_ALL_FOLDS = False    # Set True to download all 5 fold weights

files_to_download = []

# Prepare Traces
trace_path = f"{drive_path}/data/traces/train_cot_traces.jsonl"
if DOWNLOAD_TRACES and os.path.exists(trace_path):
    files_to_download.append(trace_path)
elif DOWNLOAD_TRACES:
    print(f"Trace file not found at: {trace_path}")

# Prepare CV Best Weights
cv_best_pt = f"{drive_path}/data/models/grounding_tool_cv_best/weights/best.pt"
if DOWNLOAD_CV_BEST and os.path.exists(cv_best_pt):
    files_to_download.append(cv_best_pt)
elif DOWNLOAD_CV_BEST:
    print(f"CV best weights not found at: {cv_best_pt}")

# Prepare CV Results JSON
cv_results_json = f"{drive_path}/data/models/grounding_tool_cv_best/cv_results.json"
if DOWNLOAD_CV_BEST and os.path.exists(cv_results_json):
    files_to_download.append(cv_results_json)

# Prepare All Fold Weights
if DOWNLOAD_ALL_FOLDS:
    for fold in range(5):
        fold_pt = f"{drive_path}/data/models/cv_fold_{fold}/weights/best.pt"
        if os.path.exists(fold_pt):
            files_to_download.append(fold_pt)
        else:
            print(f"Fold {fold} weights not found at: {fold_pt}")

# Download via Colab files API
try:
    from google.colab import files
    for file_path in files_to_download:
        print(f"Triggering download for: {file_path}")
        files.download(file_path)
except ImportError:
    print("Colab files utility not available. Files located at:")
    for f in files_to_download:
        print(" -", f)

## 7. Push Traces & Models to GitHub
Configures git authentication using `GITHUB_TOKEN` from secrets and pushes newly generated traces and models to GitHub.

In [ ]:
import os
import shutil
import subprocess

# Options
PUSH_TRACES = True
PUSH_MODELS = False  # Set to True if model weights are within Git repository size limits or Git LFS is configured
COMMIT_MESSAGE = "Update generated CoT traces and trained YOLO grounding models"

gh_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')

if not gh_token or gh_token.startswith('YOUR_'):
    print("ERROR: GITHUB_TOKEN is not set. Add GITHUB_TOKEN to your Colab Secrets tab or environment variables.")
else:
    git_user = os.environ.get('GIT_USER_NAME', 'rezaxr14')
    git_email = os.environ.get('GIT_USER_EMAIL', 'rezaxr14@users.noreply.github.com')
    
    # Configure Git credentials
    subprocess.run(["git", "config", "--global", "user.name", git_user])
    subprocess.run(["git", "config", "--global", "user.email", git_email])
    
    # Set Remote URL with Token
    remote_url = f"https://{gh_token}@github.com/rezaxr14/VLM-DENTAL.git"
    subprocess.run(["git", "remote", "set-url", "origin", remote_url])
    
    # Copy generated traces to repository path
    drive_trace = f"{drive_path}/data/traces/train_cot_traces.jsonl"
    repo_trace = "data/traces/train_cot_traces.jsonl"
    
    if os.path.exists(drive_trace) and os.path.abspath(drive_trace) != os.path.abspath(repo_trace):
        os.makedirs(os.path.dirname(repo_trace), exist_ok=True)
        shutil.copy(drive_trace, repo_trace)
    
    if PUSH_TRACES and os.path.exists(repo_trace):
        subprocess.run(["git", "add", "data/traces/train_cot_traces.jsonl"])
    
    if PUSH_MODELS:
        cv_best_pt = f"{drive_path}/data/models/grounding_tool_cv_best/weights/best.pt"
        if os.path.exists(cv_best_pt):
            os.makedirs("data/models/grounding_tool_cv_best/weights", exist_ok=True)
            shutil.copy(cv_best_pt, "data/models/grounding_tool_cv_best/weights/best.pt")
            subprocess.run(["git", "add", "data/models/grounding_tool_cv_best/weights/best.pt"])
        
        cv_results = f"{drive_path}/data/models/grounding_tool_cv_best/cv_results.json"
        if os.path.exists(cv_results):
            os.makedirs("data/models/grounding_tool_cv_best", exist_ok=True)
            shutil.copy(cv_results, "data/models/grounding_tool_cv_best/cv_results.json")
            subprocess.run(["git", "add", "data/models/grounding_tool_cv_best/cv_results.json"])

    # Check status and commit
    status_res = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if status_res.stdout.strip():
        subprocess.run(["git", "commit", "-m", COMMIT_MESSAGE])
        print("Pushing changes to GitHub repository...")
        subprocess.run(["git", "push", "origin", "main"])
        print("GitHub Push Complete!")
    else:
        print("No new changes detected to commit.")